# Supplementary Simulation Figure S2 — noise, heterogeneity, and fitted K

Reproduces the finalized six-panel robustness figure:
top row = spatial recovery; bottom row = temporal recovery.
Columns vary observation noise, between-subject heterogeneity, and fitted model
order.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_CSV = Path(
    "simulation_pca_msaa_extended_outputs/"
    "pca_msaa_extended_results_paper.csv"
)

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RESULTS_CSV}. Run revision_16_pca_msaa_clean_simulation_benchmark "
        "with RUN_MODE='paper' first."
    )

results = pd.read_csv(RESULTS_CSV)

METHOD_COLORS = {
    "MS-AA": "#1f77b4",
    "PCA": "#ff7f0e",
}

def mean_sem(df, group_col, value_col):
    return (
        df.groupby(group_col)[value_col]
        .agg(["mean", "sem"])
        .reset_index()
    )

FIG_PNG = OUTPUT_DIR / "supp_simulation_robustness.png"
FIG_PDF = OUTPUT_DIR / "supp_simulation_robustness.pdf"

In [ ]:
noise = results[
    results["regime"].astype(str) == "metastable_noise"
].copy()

hetero = results[
    results["regime"].astype(str) == "metastable_heterogeneity"
].copy()

kdf = results[
    results["regime"].astype(str) == "metastable_K"
].copy()

fig, axes = plt.subplots(
    2,
    3,
    figsize=(13.2, 8.0),
    gridspec_kw={"hspace": 0.34, "wspace": 0.28},
)

# A/D — noise
for row, (aa_col, pca_col, ylabel) in enumerate([
    ("aa_motif_recovery", "pca_motif_recovery", "Spatial recovery (r)"),
    ("aa_temporal_recovery", "pca_temporal_recovery", "Temporal recovery (r)"),
]):
    ax = axes[row, 0]

    for method, col in [("MS-AA", aa_col), ("PCA", pca_col)]:
        g = (
            noise.groupby("noise_sd")[col]
            .agg(["mean", "sem"])
            .reset_index()
            .sort_values("noise_sd")
        )
        ax.errorbar(
            g["noise_sd"],
            g["mean"],
            yerr=g["sem"],
            marker="o",
            linewidth=1.8,
            capsize=3,
            color=METHOD_COLORS[method],
            label=method,
        )

    ax.set_xlabel("Observation-noise SD")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)

# B/E — heterogeneity
for row, (aa_col, pca_col, ylabel) in enumerate([
    ("aa_motif_recovery", "pca_motif_recovery", "Spatial recovery (r)"),
    ("aa_temporal_recovery", "pca_temporal_recovery", "Temporal recovery (r)"),
]):
    ax = axes[row, 1]

    for method, col in [("MS-AA", aa_col), ("PCA", pca_col)]:
        g = (
            hetero.groupby("heterogeneity_scale")[col]
            .agg(["mean", "sem"])
            .reset_index()
            .sort_values("heterogeneity_scale")
        )
        ax.errorbar(
            g["heterogeneity_scale"],
            g["mean"],
            yerr=g["sem"],
            marker="o",
            linewidth=1.8,
            capsize=3,
            color=METHOD_COLORS[method],
            label=method,
        )

    ax.set_xlabel("Between-subject heterogeneity")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)

# C/F — fitted K, coverage-adjusted
for row, (aa_col, pca_col, ylabel) in enumerate([
    (
        "aa_motif_recovery_coverage",
        "pca_motif_recovery_coverage",
        "Coverage-adjusted spatial recovery",
    ),
    (
        "aa_temporal_recovery_coverage",
        "pca_temporal_recovery_coverage",
        "Coverage-adjusted temporal recovery",
    ),
]):
    ax = axes[row, 2]

    for method, col in [("MS-AA", aa_col), ("PCA", pca_col)]:
        g = (
            kdf.groupby("K_fit")[col]
            .agg(["mean", "sem"])
            .reset_index()
            .sort_values("K_fit")
        )
        ax.errorbar(
            g["K_fit"],
            g["mean"],
            yerr=g["sem"],
            marker="o",
            linewidth=1.8,
            capsize=3,
            color=METHOD_COLORS[method],
            label=method,
        )

    ax.axvline(6, color="0.55", linestyle="--", linewidth=1)
    ax.set_xlabel("Fitted model order K")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)

for ax, title in zip(
    axes[0],
    [
        "Observation noise",
        "Subject heterogeneity",
        "Fitted model order",
    ],
):
    ax.set_title(title, fontweight="bold")

for ax, letter in zip(axes.ravel(), list("ABCDEF")):
    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0, 0].legend(frameon=False)

fig.suptitle(
    "Robustness of metastable configuration recovery",
    fontsize=16,
    fontweight="bold",
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)